### 6 runnable cells in order:

| Cell                              | What it shows                                                      |
| --------------------------------- | ------------------------------------------------------------------ |
| **Sync baseline**                 | 5 links × 2s = ~10s sequential — the problem we're solving         |
| **`threading.Thread`**            | `start()`, `join()`, `daemon`, readable thread names, `is_alive()` |
| **Return-value trap**             | Why raw `Thread` discards results; shared dict + `Lock` workaround |
| **`submit()` + `as_completed()`** | `Future` as a ticket; results arrive in _completion_ order         |
| **`map()`**                       | Results in _submission_ order; when to use it vs `submit()`        |
| **Summary**                       | API comparison table, 5 interview traps                            |

---

### 📍 Module 2 — Threading Basics
#### `threading.Thread` · `.start()` · `.join()` · `ThreadPoolExecutor`

**The core mental model:**  
- `Thread.start()` → hands the thread off to the OS scheduler, returns *immediately*  
- `Thread.join()` → main thread **blocks here** until that thread finishes  
- `ThreadPoolExecutor` → same idea, but the pool manages thread lifecycle for you and gives you `Future` objects to retrieve results

[Docs](https://docs.python.org/3/library/threading.html#thread-objects)

Run the cells in order — each one builds on the previous.

In [2]:
from threading import Thread

t = Thread(target=print, args=[1])
t.run()

1


In [3]:
t = Thread(target=print, args=(1,))
t.run()

1


In [2]:
import time


# ── Shared worker function used across all examples ──────────────────────────


def crawl(link: str, delay: float = 2.0) -> str:
    """Simulates a network request (I/O-bound work)."""
    time.sleep(delay)
    return f"<html from {link}>"


LINKS = [
    "https://python.org",
    "https://docs.python.org",
    "https://peps.python.org",
    "https://pypi.org",
    "https://realpython.com",
]

In [3]:
# ── BASELINE: sequential / sync ───────────────────────────────────────────────
print("=== SYNC (sequential) ===")
start = time.perf_counter()

for link in LINKS:
    result = crawl(link)
    print(f"  done: {link}")

sync_elapsed = time.perf_counter() - start
print(f"\n⏱  Sync total: {sync_elapsed:.2f}s  (expected ≈ {len(LINKS) * 2}s — one by one)")

=== SYNC (sequential) ===
  done: https://python.org
  done: https://docs.python.org
  done: https://peps.python.org
  done: https://pypi.org
  done: https://realpython.com

⏱  Sync total: 10.00s  (expected ≈ 10s — one by one)


### `threading.Thread`: the manual approach

In [ ]:
import threading
import time

# ── threading.Thread: the manual approach ─────────────────────────────────────
#
# Thread(target, args, kwargs, name, daemon)
#   target  → callable to run in the new thread
#   args    → positional args tuple
#   kwargs  → keyword args dict
#   name    → readable label (shows up in logs and debuggers)
#   daemon  → if True, thread is killed automatically when the main thread exits
#             (default False: main thread waits for ALL non-daemon threads)

print("=== threading.Thread ===\n")
start = time.perf_counter()

threads = []
for link in LINKS:
    t = threading.Thread(
        target=crawl,
        args=(link,),
        kwargs={"delay": 2.0},
        name=f"Crawler-{link.split('//')[1][:12]}",
        daemon=True,
    )
    threads.append(t)

# .start() → schedules the thread with the OS, returns immediately
# At this point all 5 threads are alive and sleeping concurrently
for t in threads:
    t.start()
    print(f"  started  [{t.name}]  alive={t.is_alive()}")

print()

# .join() → main thread blocks HERE until the thread whose join() method is called terminates
# Without join(), main could exit before threads complete
for t in threads:
    t.join()
    print(f"  joined   [{t.name}]  alive={t.is_alive()}")

threaded_elapsed = time.perf_counter() - start
print(f"\n⏱  Threaded total : {threaded_elapsed:.2f}s  (expected ≈ 2s — all 5 overlap)")
print(f"⚡ Speedup vs sync: {sync_elapsed / threaded_elapsed:.1f}x")

=== threading.Thread ===

  started  [Crawler-python.org]  alive=True
  started  [Crawler-docs.python.]  alive=True
  started  [Crawler-peps.python.]  alive=True
  started  [Crawler-pypi.org]  alive=True
  started  [Crawler-realpython.c]  alive=True

  joined   [Crawler-python.org]  alive=False
  joined   [Crawler-docs.python.]  alive=False
  joined   [Crawler-peps.python.]  alive=False
  joined   [Crawler-pypi.org]  alive=False
  joined   [Crawler-realpython.c]  alive=False

⏱  Threaded total : 2.00s  (expected ≈ 2s — all 5 overlap)
⚡ Speedup vs sync: 5.0x


1. **The module runs in the main process**
    
    - When you run the Python file, the interpreter starts **one process**.
    - The code that is executing is the **main thread** of that process.
2. **`threading.Thread(...)` creates thread objects**
    
    - This does **not** start the work yet.
    - It just creates a `Thread` object that will run the target function when started.
3. **`t.start()` launches a new thread**
    
    - Each `t.start()` tells the OS to schedule a new thread.
    - That new thread runs `crawl(...)` independently.
    - The main thread continues immediately to the next `t.start()` call.
4. **The worker threads are concurrent**
    
    - In your example, each thread is sleeping for 2 seconds.
    - So they are effectively all running “at the same time” from the program’s perspective.
5. **`t.join()` blocks the main thread**
    
    - When the main thread reaches `t.join()`, it waits until **that specific thread** finishes.
    - So:
        - first `join()` waits for thread 1
        - second `join()` waits for thread 2
        - etc.
    - The main thread does **not** wait for all threads at once; it waits **one by one** in the order of the `join()` calls.

**Important nuance**

- The threads are **not** separate processes.
- They are **threads inside the same process**.
- The main thread is still part of that same process.

---

### ⚠️ INTERVIEW TRAP
**`Thread.target` return values are silently discarded**

In [5]:
import threading
import time

# ── ⚠️ INTERVIEW TRAP: Thread.target return values are silently discarded ──────
#
# threading.Thread has no built-in way to retrieve what target() returned.
# The naive fix: write results into a shared dict (protected by a Lock).

results: dict[str, str] = {}
lock = threading.Lock()

def crawl_and_store(link: str, delay: float = 2.0) -> None:
    time.sleep(delay)
    html = f"<html from {link}>"
    with lock:                  # only one thread writes at a time
        results[link] = html

In [6]:
threads = [
    threading.Thread(target=crawl_and_store, args=(link,))
    for link in LINKS
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Collected results (shared dict pattern):")
for url, html in results.items():
    print(f"  {url!r:45s} → {html}")

print()
print("NOTE: ThreadPoolExecutor (next cell) solves this cleanly via Future.result()")

Collected results (shared dict pattern):
  'https://python.org'                          → <html from https://python.org>
  'https://pypi.org'                            → <html from https://pypi.org>
  'https://peps.python.org'                     → <html from https://peps.python.org>
  'https://realpython.com'                      → <html from https://realpython.com>
  'https://docs.python.org'                     → <html from https://docs.python.org>

NOTE: ThreadPoolExecutor (next cell) solves this cleanly via Future.result()


### `ThreadPoolExecutor`: `submit()` + Future + `as_completed()`

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# ── ThreadPoolExecutor: submit() + Future + as_completed() ────────────────────
#
# submit(fn, *args, **kwargs)
#   → schedules fn(*args, **kwargs) on a worker thread
#   → returns a Future *immediately* (non-blocking)
#   → Future is a "ticket": call .result() later to collect the return value
#
# as_completed(futures)
#   → yields each Future IN THE ORDER IT FINISHES (fastest first)
#   → useful when you want to process results as soon as they arrive

print("=== ThreadPoolExecutor: submit() + as_completed() ===\n")
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as executor:
    # Map each Future back to its URL so we know which result belongs where
    future_to_url: dict = {
        executor.submit(crawl, link, 2.0): link    # {Future: url-link}
        for link in LINKS
    }

    for future in as_completed(future_to_url):
        url = future_to_url[future]
        try:
            result = future.result()   # raises if the worker raised an exception
            print(f"  ✓ finished first: {url}")
            print(f"           result: {result}")
        except Exception as exc:
            print(f"  ✗ {url} raised: {exc}")

# The `with` block exits only after ALL submitted futures are done (implicit join)
tpe_submit_elapsed = time.perf_counter() - start
print(f"\n⏱  TPE submit+as_completed: {tpe_submit_elapsed:.2f}s")
print("   (results printed in completion order, NOT submission order)")


=== ThreadPoolExecutor: submit() + as_completed() ===

  ✓ finished first: https://python.org
           result: <html from https://python.org>
  ✓ finished first: https://docs.python.org
           result: <html from https://docs.python.org>
  ✓ finished first: https://peps.python.org
           result: <html from https://peps.python.org>
  ✓ finished first: https://realpython.com
           result: <html from https://realpython.com>
  ✓ finished first: https://pypi.org
           result: <html from https://pypi.org>

⏱  TPE submit+as_completed: 2.00s
   (results printed in completion order, NOT submission order)


In [9]:
from concurrent.futures import ThreadPoolExecutor
import time

# ── ThreadPoolExecutor: map() ─────────────────────────────────────────────────
#
# executor.map(fn, iterable)
#   → submits fn(item) for each item, concurrently
#   → returns an iterator that yields results IN SUBMISSION ORDER
#   → blocks on each result as you iterate (but all workers run concurrently)
#   → raises immediately if any worker raised (unlike as_completed where you
#     handle exceptions per-future)
#
# Use map() when:  order matters AND you don't need per-result error handling
# Use submit()  when: you want results as they arrive OR per-future error handling

print("=== ThreadPoolExecutor: map() ===\n")
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as executor:
    # map() with multiple iterables: fn(link, delay) for each pair
    results = list(executor.map(crawl, LINKS, [2.0] * len(LINKS)))

tpe_map_elapsed = time.perf_counter() - start
print(f"⏱  TPE map: {tpe_map_elapsed:.2f}s")
print("Results in ORIGINAL submission order (same as LINKS list):\n")
for url, result in zip(LINKS, results):
    print(f"  {url!r:45s} → {result}")

=== ThreadPoolExecutor: map() ===

⏱  TPE map: 2.00s
Results in ORIGINAL submission order (same as LINKS list):

  'https://python.org'                          → <html from https://python.org>
  'https://docs.python.org'                     → <html from https://docs.python.org>
  'https://peps.python.org'                     → <html from https://peps.python.org>
  'https://pypi.org'                            → <html from https://pypi.org>
  'https://realpython.com'                      → <html from https://realpython.com>


## Summary & Interview Traps

### API comparison

| | `threading.Thread` | `TPE submit()` + `as_completed()` | `TPE map()` |
|---|---|---|---|
| **Result access** | ❌ no built-in — write to shared dict | ✅ `future.result()` per future | ✅ iterator in order |
| **Result order** | whatever you wrote | completion order (fastest first) | submission order |
| **Error handling** | exception is lost unless you catch inside `target` | `future.result()` re-raises | first exception stops iteration |
| **Thread lifecycle** | manual `start()` + `join()` | pool manages it; `with` block joins all | same |
| **When to choose** | fine-grained control, daemon threads, custom names | process results as they arrive | order matters, simple mapping |

---

### ⚠️ Interview traps on this topic

1. **`Thread` discards return values.**  
   `t = Thread(target=fn); t.start(); t.join()` — you cannot call `t.result()`. Use TPE or a shared container.

2. **`t.start()` vs `t.run()`**  
   `t.run()` executes the target *in the calling thread* — no new thread is created. It's almost always a bug.

3. **`join()` timeout ≠ cancellation.**  
   `t.join(timeout=1)` returns after 1s *whether or not the thread finished*. Check `t.is_alive()` after to know which happened.

4. **`daemon=True` threads are killed abruptly.**  
   If the main thread exits, all daemon threads are killed immediately — no cleanup, no `finally` blocks.

5. **`as_completed` order is non-deterministic.**  
   Never assume futures come back in the order they were submitted.

---

### ✅ Check question

> You call `executor.submit(fetch_url, url)` for 100 URLs with `max_workers=10`.  
> Inside `fetch_url` an exception is raised for URL #50.  
> **What happens?** Does the pool stop? Do other futures raise? How do you detect and handle the error?

Answer below before moving on ↓